# Compact Convolutional Autoencoder for STL-10

Compact convolutional autoencoder for STL-10 (96x96x3 RGB natural
images). Same architecture pattern as the MNIST autoencoder
(`code/mnist_conv_autoencoder.py`) -- explicit strided-conv encoder,
FC bottleneck, transposed-conv decoder -- scaled up with one extra
conv/deconv stage and more channels, so that reconstruction quality
(blur, texture loss, color fidelity) is much more visible than it is
on simple MNIST digits.

**Architecture** (`STL10ConvAutoencoder`, defined in
`code/stl10_conv_autoencoder.py`), shape trace for `latent_dim=128`:

```
input                              [B,  3, 96, 96]
--- encoder ---
conv1 3x3 s2 p1 (3->32)        -> [B, 32, 48, 48]
relu1                          -> [B, 32, 48, 48]
conv2 3x3 s2 p1 (32->64)       -> [B, 64, 24, 24]
relu2                          -> [B, 64, 24, 24]
conv3 3x3 s2 p1 (64->128)      -> [B,128, 12, 12]
relu3                          -> [B,128, 12, 12]
conv4 3x3 s2 p1 (128->256)     -> [B,256,  6,  6]
relu4                          -> [B,256,  6,  6]
flatten                        -> [B, 9216]
fc_enc (9216->128)             -> [B,  128]         (latent z)
--- decoder ---
fc_dec (128->9216)             -> [B, 9216]
relu_dec                       -> [B, 9216]
unflatten                      -> [B,256,  6,  6]
deconv1 3x3 s2 p1 op1 (256->128) -> [B,128, 12, 12]
relu_d1                          -> [B,128, 12, 12]
deconv2 3x3 s2 p1 op1 (128->64)  -> [B, 64, 24, 24]
relu_d2                          -> [B, 64, 24, 24]
deconv3 3x3 s2 p1 op1 (64->32)   -> [B, 32, 48, 48]
relu_d3                          -> [B, 32, 48, 48]
deconv4 3x3 s2 p1 op1 (32-> 3)   -> [B,  3, 96, 96]
sigmoid                          -> [B,  3, 96, 96]   (x_hat in [0,1])
```

96 is divisible by 16, so every encoder halving (96/48/24/12/6) is
exact with no odd-size rounding (unlike MNIST's 28->14->7->4, where 7
is odd) -- that's why every decoder stage here uses the same
`output_padding=1`, instead of the mixed 0/1 pattern in the MNIST
model. See the class docstring in the `.py` file for the full
derivation.

The model is trained to minimize pixel-wise reconstruction MSE,
`L(x) = mean((x - decoder(encoder(x)))^2)`, with no label supervision.

**Note on scale**: STL-10 is a ~2.6GB download (vs. MNIST's ~12MB), so
this notebook *must* run on Colab with GPU + Drive caching -- do not
attempt real training locally.

Core code lives in `code/stl10_conv_autoencoder.py` (version-controlled);
this notebook is a thin orchestration layer: GPU check, repo pull, build
model, print shape trace + parameter count, download real STL-10
(cached in Google Drive so it is downloaded only once), train, plot the
loss curve, and show original-vs-reconstruction qualitative +
quantitative results (per-image MSE) as the required results-
visualization step.

## 1. GPU check

In [ ]:
import torch

# Never hard-code CUDA: fall back to CPU if no GPU is available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))


## 2. Clone/pull repository

Model code lives in `code/stl10_conv_autoencoder.py`, not duplicated in
this notebook. Replace `REPO_URL` with this repository's URL.

In [ ]:
import os

REPO_URL = "https://github.com/RoeiZ/from_zero_to_hero.git"
REPO_DIR = "/content/from_zero_to_hero"

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "code"))


## 3. Build model, print shape trace and parameter count

One verbose forward pass on a small synthetic batch, purely to print
the `[B,C,H,W]` / `[B,D]` shape after every encoder/decoder op. Not
part of training.

In [ ]:
from stl10_conv_autoencoder import STL10ConvAutoencoder, count_trainable_parameters

torch.manual_seed(0)

LATENT_DIM = 128

model = STL10ConvAutoencoder(latent_dim=LATENT_DIM).to(device)
print(f"trainable parameters: {count_trainable_parameters(model):,}")

sample_batch = torch.rand(4, 3, 96, 96, device=device)  # [B,3,96,96], synthetic, in [0,1]
_ = model(sample_batch, verbose=True)


## 4. Download real STL-10 (train/test split), cached in Drive

`torchvision.datasets.STL10(split="train")` / `split="test"` gives the
dataset's native labeled train/test partition (the separate 100k-image
`unlabeled` split is not used, since an autoencoder needs no labels but
the labeled splits already give a clean train/test partition). No extra
validation split is invented here, matching the convention used for
MNIST/CIFAR-10 in this repo.

Colab's local disk (`/content/...`) is wiped whenever the runtime
recycles, so this ~2.6GB dataset would otherwise be re-downloaded on
every fresh session. Instead we mount Google Drive and point
`DATA_ROOT` at a persistent Drive folder: `torchvision` only downloads
into it once and simply reads from it on every later run.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Persistent cache: downloaded once into Drive, reused on every
# future Colab session (local /content storage is ephemeral).
DATA_ROOT = "/content/drive/MyDrive/thesis_data/stl10"


In [ ]:
from stl10_conv_autoencoder import get_stl10_dataloaders

BATCH_SIZE = 64  # smaller than MNIST/CIFAR-10 batch size: 96x96x3 images use much more GPU memory per sample

train_loader, test_loader = get_stl10_dataloaders(DATA_ROOT, BATCH_SIZE)
print(f"train batches = {len(train_loader)}, test batches = {len(test_loader)}")


## 5. Train

Explicit train/eval loops (`train_one_epoch`, `evaluate`), MSE
reconstruction loss (image vs. its own reconstruction, no labels used),
Adam optimizer.

In [ ]:
import torch.nn as nn
from stl10_conv_autoencoder import train_one_epoch, evaluate

N_EPOCHS = 15
LR = 1e-3

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = {"train_mse": [], "test_mse": []}

for epoch in range(N_EPOCHS):
    train_mse = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    test_mse = evaluate(model, test_loader, loss_fn, device)

    history["train_mse"].append(train_mse)
    history["test_mse"].append(test_mse)

    print(f"epoch {epoch + 1:3d}/{N_EPOCHS}  train_mse={train_mse:.4f}  test_mse={test_mse:.4f}")


## 6. Reconstruction-loss curve (quantitative result over training)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(history["train_mse"], label="train")
ax.plot(history["test_mse"], label="test")
ax.set_xlabel("epoch")
ax.set_ylabel("pixel-wise MSE")
ax.set_title("Reconstruction loss")
ax.legend()
fig.tight_layout()
plt.show()

print(f"final test_mse = {history['test_mse'][-1]:.4f}")


## 7. Qualitative + quantitative check: original vs. reconstruction

Per-project convention: always look at concrete examples, not only
scalar metrics. Show 5 real test images (top row) next to the model's
reconstruction (bottom row), with the per-image reconstruction MSE,
PSNR, and SSIM (structural similarity, Wang et al. 2004 -- see the
`ssim` function in the `.py` file) printed under each reconstruction
-- the quantitative results shown with respect to the original.
STL-10's higher resolution and real textures/colors make blur and
detail loss far easier to see here than on MNIST.

In [ ]:
from stl10_conv_autoencoder import visualize_reconstructions

sample_images, _sample_labels = next(iter(test_loader))
sample_images = sample_images.to(device)

visualize_reconstructions(model, sample_images, num_examples=5)


## 8. Test-set average reconstruction MSE

A single aggregate number (over the full test split, not just the 5
plotted examples) for comparison across runs/configs.

In [ ]:
final_test_mse = evaluate(model, test_loader, loss_fn, device)
print(f"average test-set reconstruction MSE (latent_dim={LATENT_DIM}): {final_test_mse:.4f}")


## 9. Compression-ratio comparison: does a smaller bottleneck hurt quality?

Trains a **separate** `STL10ConvAutoencoder` from scratch for each
`latent_dim` in `LATENT_DIMS` (they have differently-shaped `fc_enc`/
`fc_dec` layers, so weights can't be shared), then shows 2 real test
images reconstructed by each one side by side, with per-image MSE,
**PSNR** (`10*log10(1/MSE)`, in dB), and **SSIM** (structural
similarity -- unlike MSE/PSNR, which are purely pixel-wise, SSIM
compares local luminance/contrast/structure, so it can catch cases
where two reconstructions tie on MSE but look very different, e.g.
blur vs. noise) under each reconstruction.
`compression_ratio = (3*96*96) / latent_dim` -- how many fewer numbers
the bottleneck stores than the raw image.

This trains `len(LATENT_DIMS)` models, each for `COMPRESSION_EPOCHS`
epochs (kept short here just to show the *trend* across compression
levels, not to fully converge each one) -- real GPU compute, Colab
only. Uses the same `train_loader`/`test_loader` already built above.

In [ ]:
from stl10_conv_autoencoder import run_compression_ratio_comparison

LATENT_DIMS = (16, 64, 256)   # small / medium / mild compression
COMPRESSION_EPOCHS = 5

models_by_latent_dim, compression_summary = run_compression_ratio_comparison(
    DATA_ROOT,
    batch_size=BATCH_SIZE,
    epochs=COMPRESSION_EPOCHS,
    latent_dims=LATENT_DIMS,
    num_example_images=2,
    device=device,
)
